## Transforming the AD BEL KG to a CellDesigner AF mapas an ordinary CellDesigner collection

We transform the influence projection of the AD BEL KG into a CellDesigner map, with layout elements (but no placement for them).
The transformation has the following specificities:
* `act(X)` is transformed into an active `X` species;
* isolated species are dropped;
* both biological pathways and pathology elements are transformed into phenotypes.

In [1]:
%store -r

In [2]:
import pathlib

import commute_dm.bel_submaps
import commute_dm.bel_terms
import commute_dm.core
import commute_dm.queries
import commute_dm.submaps
import credentials
import momapy.io.core
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Parameters

`CD_AF_SOURCE_COLLECTION_NAMES` are the collections whose stored elements the integration cache is
seeded from, so that an element this export shares with them becomes **one** database node rather
than a duplicate. All four CellDesigner collections are listed, not just the AF ones: the AF maps
share elements with the process-description maps they were derived from.

In [4]:
BEL_COLLECTION_NAME = "AD_KG_BEL"
CD_COLLECTION_NAME = "AD_KG_CD_AF"
CD_AF_SOURCE_COLLECTION_NAMES = [
    "COVID_DM_CD",
    "COVID_DM_CD_AF",
    "PD_DM_CD",
    "PD_DM_CD_AF",
]
OUTPUT_FILE_PATH = AD_KG_CD_AF_BUILD_DIR / "ad_kg.xml"
OUTPUT_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE_PATH

PosixPath('../../build/maps/ad_kg/celldesigner_af/ad_kg.xml')

## Build the map (~2 s)

`load_bel_projection` gives the projected nodes and their signed causal edges, `load_bel_terms` the
term graph, and `make_bel_map` turns the two into a `CellDesignerMap` of real activity-flow content:
proteins with phosphorylation badges, complexes with nested subunit glyphs, `act(...)` as the active
decoration, `loc(...)` as a drawn compartment.

No global auto-layout is run. `pd2af.utils.make_auto_layout` shells out to graphviz `dot`, which does
not finish on a graph this size; the glyphs keep local, non-overlapping positions, and the sub-maps
of `4_10` lay out their own (much smaller) selections.

In [5]:
nodes, edges = commute_dm.bel_submaps.load_bel_projection(
    session, [BEL_COLLECTION_NAME]
)
terms = commute_dm.bel_terms.load_bel_terms(session, [BEL_COLLECTION_NAME])
cd_map, species_by_node_id, node_id_species_pairs, stats = (
    commute_dm.bel_submaps.make_bel_map(nodes, terms, edges)
)
for key in sorted(stats):
    print(f"{key:40s} {stats[key]}")

max_complex_depth                        2
n_active                                 436
n_compartment_roots                      1
n_compartments                           4
n_drawn_compartments                     4
n_isolated_species_dropped               1175
n_mapping_entries                        3555
n_modifications                          206
n_modulations                            4631
n_named_by_suffix                        11
n_node_id_species_pairs                  4374
n_node_ids_kept                          2803
n_projected_node_ids                     3979
n_self_loop_edges_dropped                1
n_species                                2418
n_species_collapsed                      385
n_structural_states                      148
n_subunits                               779
n_templates                              1738
n_templates_interned_from_source_map     0
n_terms_described                        4513


Expected: **2418** species (of 3593 before the drop, so **1175** isolated species dropped, standing
for 1228 isolated node ids), **385** node ids collapsed by interning, **436** active species, **779**
subunits, **206** modifications, **148** structural states, **4631** modulations, **1206** templates
and **5** compartments (4 drawn `loc()` boxes plus the undrawn per-collection root).

## Annotations

The exported species carry the cross-references of the BEL nodes they stand for, which is what puts
this collection's proteins in the interface. `get_annotated_nodes` decides whose annotations stand
for a node: an activity's subject is followed, a complex's members are **not** -- each member is
drawn as its own subunit species and annotated in its own right.

Annotating subunits is not optional: **16 of the interface's UniProt identifiers come only from
complex members**. Without them the two-way interface is 173 rather than 189, and the three-way 116
rather than 127.

In [6]:
commute_dm.queries.prewarm_session(session)
element_to_annotations = commute_dm.bel_submaps.make_element_to_annotations(
    session, node_id_species_pairs
)
uniprot_resources = {
    resource
    for annotations in element_to_annotations.values()
    for annotation in annotations
    for resource in annotation.resources
    if "uniprot" in resource
}
print("annotated species        ", len(element_to_annotations))
print("distinct uniprot ids     ", len(uniprot_resources))

annotated species         954
distinct uniprot ids      778


Expected: **954** annotated species carrying **778** distinct UniProt ids.

## Renumber, then write (~10 s, ~16 MB)

`renumber_ids` is **mandatory**, not tidying. Without it 1778 of the 1782 subunit glyphs are silently
lost on read-back: momapy writes an alias id SBML-sanitised and its back-reference raw, and a uuid4
`id_` contains `-`. (Worth reporting upstream; nothing here needs the fix.)

It round-trips every object through `momapy.builder`, so the annotation dict -- keyed on the
pre-renumber objects -- has to be re-keyed onto the rebuilt ones. Equality finds the match; doing it
explicitly means the writer is never handed a stale object.

In [7]:
cd_map = commute_dm.submaps.renumber_ids(cd_map)
element_to_annotations = {
    species: annotations
    for species in commute_dm.submaps.iter_species_and_subunits(cd_map.model.species)
    if (annotations := element_to_annotations.get(species))
}
commute_dm.submaps.check_identity_invariants(cd_map)
_ = momapy.io.core.write(
    cd_map,
    OUTPUT_FILE_PATH,
    writer="celldesigner",
    element_to_annotations=element_to_annotations,
)
print(f"{OUTPUT_FILE_PATH} ({OUTPUT_FILE_PATH.stat().st_size / 1e6:.1f} MB)")

../../build/maps/ad_kg/celldesigner_af/ad_kg.xml (16.3 MB)


## Read back and check (~5 s)

The counts of the map read back from the file must equal the in-memory map's, and the four identity
invariants must hold on it. Getting one of those wrong produces a file that writes without error and
fails to read back with a `KeyError`, which is why the check is here rather than assumed.

A `Modification` whose `state` is `None` is **ignored** in the comparison: the reader fills one in
for every template residue a species does not carry, which is CellDesigner's own proteoform
semantics, not a round-trip defect.

In [8]:
def count_map(a_map):
    all_species = list(
        commute_dm.submaps.iter_species_and_subunits(a_map.model.species)
    )
    return {
        "species": len(a_map.model.species),
        "subunits": len(all_species) - len(a_map.model.species),
        "templates": len(a_map.model.species_templates),
        "compartments": len(a_map.model.compartments),
        "modulations": len(a_map.model.modulations),
        "active": sum(bool(species.active) for species in a_map.model.species),
        # `state is None` is the reader's own fill-in; see above.
        "modifications": sum(
            1
            for species in all_species
            for modification in (getattr(species, "modifications", ()) or ())
            if modification.state is not None
        ),
        "structural_states": sum(
            len(getattr(species, "structural_states", ()) or ())
            for species in all_species
        ),
    }


result = momapy.io.core.read(OUTPUT_FILE_PATH, reader="celldesigner")
written_counts = count_map(cd_map)
read_counts = count_map(result.obj)
for key in written_counts:
    status = "OK" if written_counts[key] == read_counts[key] else "DIFFERENT"
    print(
        f"{key:20s} written={written_counts[key]:6d} read={read_counts[key]:6d}  {status}"
    )
assert written_counts == read_counts, "the map does not survive a round trip"
commute_dm.submaps.check_identity_invariants(result.obj)

read_uniprot_resources = {
    resource
    for annotations in result.element_to_annotations.values()
    for annotation in annotations
    for resource in annotation.resources
    if "uniprot" in resource
}
assert read_uniprot_resources == uniprot_resources, (
    "uniprot annotations did not survive"
)
print(
    f"\nidentity invariants pass; {len(read_uniprot_resources)} uniprot ids round-tripped"
)

species              written=  2418 read=  2418  OK
subunits             written=   779 read=   779  OK
templates            written=  1206 read=  1206  OK
compartments         written=     5 read=     5  OK
modulations          written=  4631 read=  4631  OK
active               written=   436 read=   436  OK
modifications        written=   206 read=   206  OK
structural_states    written=   148 read=   148  OK

identity invariants pass; 778 uniprot ids round-tripped


## Guard

The save below is a plain `CREATE`. Running it twice would silently give two copies of every element
of this collection, so the notebook refuses to go on if the collection already exists. Drop it first
if you mean to re-import:

```cypher
MATCH (c:Collection {name: 'AD_KG_CD_AF'})-[:HAS_ENTRY]->(e)-[:HAS_OBJ]->(m)
DETACH DELETE c, e, m
```

(that leaves the model elements themselves behind, which the next import will simply re-integrate).

In [9]:
existing = session.execute_query(
    "MATCH (collection:Collection {name: $name}) RETURN count(collection) AS n",
    params={"name": CD_COLLECTION_NAME},
)
assert existing[0]["n"] == 0, (
    f"collection {CD_COLLECTION_NAME} already exists; the save below is a plain CREATE "
    "and would double it"
)
print(f"{CD_COLLECTION_NAME} does not exist yet")

AD_KG_CD_AF does not exist yet


## Seed the integration cache (~4 s)

Collections are imported with `integration_mode="hash"`, which is meant to make two content-equal
elements **one** database node. That only held *within one save call*: `save_from_objects` built its
`object_to_node` map fresh and never read the database. This export is a second save call, so
without the seeding below it would create a duplicate node for every element it shares with the
stored maps -- and duplicates break the writer, since `CellDesignerModel` holds its elements in
frozensets while the writer resolves `<proteinReference>` by object identity.

`execute_query_as_objects` fills `object_key_to_node`, and
`save_collections_from_file_paths` consumes it, so seeding it from the database makes hash
integration span calls.

**The seed set has to be closed under descent.** A cache hit skips the walk over that object's own
descendants, so anything under a seeded element would get no cache entry and hence no
`HAS_MODEL_ELEMENT` edge from this collection's model -- the edge every helper in `commute_dm.queries`
runs on. This set is closed, because it *is* `model.descendants()`.

In [10]:
SEED_QUERY = """
MATCH (c:Collection)-[:HAS_ENTRY]->()-[:HAS_OBJ]->(:CellDesignerMap)
      -[:HAS_MODEL]->(:Model)-[:HAS_MODEL_ELEMENT]->(e)
WHERE c.name IN $names
RETURN DISTINCT e
"""

object_key_to_node = {}
_ = session.execute_query_as_objects(
    SEED_QUERY,
    params={"names": CD_AF_SOURCE_COLLECTION_NAMES},
    node_id_to_object={},
    object_key_to_node=object_key_to_node,
)
print("integration cache entries", len(object_key_to_node))

integration cache entries 31852


Expected: **31852** entries.

## Save

`2_00`'s call, for this collection only, plus the seeded cache. `with_membership_edges=True` is what
gives every model element its `HAS_MODEL_ELEMENT` edge, which `get_collections_for_nodes` and the
interface query rely on.

In [11]:
_ = session.save_collections_from_file_paths(
    [(CD_COLLECTION_NAME, [OUTPUT_FILE_PATH])],
    return_type="map",
    with_membership_edges=True,
    integration_mode="hash",
    object_key_to_node=object_key_to_node,
)
print("saved", CD_COLLECTION_NAME)

saved AD_KG_CD_AF


## Post-import assertions

Three things are checked: the collection holds what the file holds; every model element has its
membership edge; and the elements this export shares with the stored collections became *one* node
rather than two.

The fusion counts are the point of the seeding. Expected: **242** templates, **88** top-level
species, **41** subunits and **0** compartments or modulations shared with the CellDesigner
collections. All 88 fused top-level species are stored **only as complex subunits** -- a stored
top-level species carries a compartment and a BEL one does not, so those two can never be
value-equal -- which is the case `submaps.make_submap_from_model_elements`' subunit promotion
already covers.

In [12]:
counts = session.execute_query(
    """
    MATCH (c:Collection {name: $name})-[:HAS_ENTRY]->()-[:HAS_OBJ]->(:CellDesignerMap)
          -[:HAS_MODEL]->(model:Model)
    MATCH (model)-[:HAS_MODEL_ELEMENT]->(e)
    RETURN labels(e) AS labels, count(*) AS n
    """,
    params={"name": CD_COLLECTION_NAME},
)
by_class = {}
for row in counts:
    for label in ("Species", "Compartment", "SpeciesTemplate"):
        if label in row["labels"]:
            by_class[label] = by_class.get(label, 0) + row["n"]
print(by_class)

{'SpeciesTemplate': 1206, 'Species': 3197, 'Compartment': 5}


In [13]:
orphans = session.execute_query(
    """
    MATCH (c:Collection {name: $name})-[:HAS_ENTRY]->()-[:HAS_OBJ]->(cd_map:CellDesignerMap)
    MATCH (cd_map)-[:HAS_MODEL]->(model:Model)
    MATCH (model)-[*1..12]->(e:ModelElement)
    WHERE NOT (model)-[:HAS_MODEL_ELEMENT]->(e)
    RETURN count(DISTINCT e) AS n
    """,
    params={"name": CD_COLLECTION_NAME},
)
print("model elements without a membership edge:", orphans[0]["n"])

model elements without a membership edge: 0


In [14]:
fused = session.execute_query(
    """
    MATCH (ad:Collection {name: $name})-[:HAS_ENTRY]->()-[:HAS_OBJ]->(:CellDesignerMap)
          -[:HAS_MODEL]->(:Model)-[:HAS_MODEL_ELEMENT]->(e)
    MATCH (other:Collection)-[:HAS_ENTRY]->()-[:HAS_OBJ]->(:CellDesignerMap)
          -[:HAS_MODEL]->(:Model)-[:HAS_MODEL_ELEMENT]->(e)
    WHERE other.name IN $others
    RETURN labels(e) AS labels, count(DISTINCT e) AS n
    """,
    params={"name": CD_COLLECTION_NAME, "others": CD_AF_SOURCE_COLLECTION_NAMES},
)
shared = {}
for row in fused:
    for label in ("Species", "Compartment", "SpeciesTemplate", "Modulation"):
        if label in row["labels"]:
            shared[label] = shared.get(label, 0) + row["n"]
print("shared with the CellDesigner collections:", shared)

shared with the CellDesigner collections: {'SpeciesTemplate': 258, 'Species': 116}


Finally the interfaces, which are what the rest of the pipeline runs on. Expected: **159**
two-way (`COVID_DM_CD_AF` x `AD_KG_CD_AF`) and **111** three-way. These are smaller than the 189 /
127 of the `AD_KG_BEL` pairing because the isolated nodes were dropped on purpose: the identifiers
they carried could never have seeded a walk, so the interface now *is* the seedable set.

In [15]:
two_way = commute_dm.core.get_interface(session, ["COVID_DM_CD_AF", CD_COLLECTION_NAME])
three_way = commute_dm.core.get_interface(
    session, ["COVID_DM_CD_AF", "PD_DM_CD_AF", CD_COLLECTION_NAME]
)
print("two-way interface  ", len(two_way))
print("three-way interface", len(three_way))

two-way interface   159
three-way interface 111


`2_05` is numbered before this notebook, so **re-run it now** to pick the new collection up: its
`CD_AF_COLLECTIONS` constant already lists `AD_KG_CD_AF`.